In [ ]:
"""Measure shallow / deep / end-to-end FPS for the VISTA pipeline.

Definitions (per the challenge):
  shallow FPS    = detection + tracking only (runs every frame)        -> floor >= 5
  deep   FPS     = VLM captioning, amortised over the caption stride    (informative)
  end-to-end FPS = full pipeline (det+track + VLM at its stride)

Method: we time ONLY the pipeline.forward() calls (video decode is excluded, since
the harness hands the pipeline an already-decoded frame). We run two passes over
the same clip:
  * pass A with the captioner disabled  -> time = detection+tracking  -> shallow
  * pass B with the full pipeline       -> time = everything          -> end-to-end
and derive the captioning time as (B - A), so:
  deep FPS = n_frames / (t_e2e - t_shallow)   # amortised: accounts for the stride

Notes / caveats:
  * Run on a REPRESENTATIVE VISTA clip: shallow FPS depends on resolution, and deep
    FPS depends on object density (Approach 1 crops every track) and on caption_stride.
  * Warmup frames are excluded (first CUDA calls are slow due to lazy kernel init).
  * torch.cuda.synchronize() ensures GPU work is finished before stopping the timer.
  * The (B - A) subtraction assumes detection time is stable across the two passes
    (true to within noise). If you want an exact single-pass split instead, add two
    perf_counter timers inside forward(): one around the yolo.track() call
    (accumulate into self._t_track) and one around the captioning block
    (self._t_caption), then shallow = n/self._t_track, deep = n/self._t_caption.

    uv run python measure_fps.py
"""



In [ ]:
import time
from pathlib import Path

import cv2
import torch
from PIL import Image

# ── config — VERIFY these ───────────────────────────────────────────────────────
VIDEO = Path("/workspace/VISTA/data/<a_representative_clip>.mp4")   # or a frame dir
CFG = "mycfg.yaml"
MAX_FRAMES = 600 # enough for a stable average (~20 caption calls at stride 30)
WARMUP = 5
FPS_FLOOR = 5 # challenge shallow-FPS floor for two-stage pipelines

# build your pipeline however you normally do; it must support captioner=None
from mypipeline import build_mypipeline_from_config   # adjust to your module/builder




In [ ]:
def _frame_iter(video_path: Path):
    """Yield RGB PIL frames; decode happens outside the timed region."""
    if video_path.is_dir():
        for p in sorted(video_path.glob("*.jpg")):
            yield Image.open(p).convert("RGB")
    else:
        cap = cv2.VideoCapture(str(video_path))
        while True:
            ok, bgr = cap.read()
            if not ok:
                break
            yield Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        cap.release()


def time_forward(pipe, video_path, warmup=WARMUP, max_frames=MAX_FRAMES):
    """Return (n_frames_timed, total_forward_seconds). Times only forward()."""
    pipe.reset()
    idx = 0
    n = 0
    total = 0.0
    for frame in _frame_iter(video_path):
        if idx < warmup:                       # warmup: run but don't time
            pipe.forward(frame, idx)
            idx += 1
            continue
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        pipe.forward(frame, idx)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        total += time.perf_counter() - t0
        n += 1
        idx += 1
        if max_frames and n >= max_frames:
            break
    return n, total

In [ ]:
pipe = build_mypipeline_from_config(CFG)
captioner = pipe.captioner            # keep a handle to restore it

# pass A — detection + tracking only
pipe.captioner = None
n_a, t_shallow = time_forward(pipe, VIDEO)
# pass B — full pipeline
pipe.captioner = captioner
n_b, t_e2e = time_forward(pipe, VIDEO)

assert n_a == n_b, f"frame counts differ ({n_a} vs {n_b}); use a fixed MAX_FRAMES"
n = n_a
t_caption = max(t_e2e - t_shallow, 1e-9)

shallow_fps = n / t_shallow
deep_fps = n / t_caption
e2e_fps = n / t_e2e

print(f"\nframes timed            : {n}")
print(f"shallow FPS (det+track) : {shallow_fps:7.1f}   "
          f"({1000 * t_shallow / n:.1f} ms/frame)   floor >= {FPS_FLOOR}: "
      f"{'PASS' if shallow_fps >= FPS_FLOOR else 'FAIL'}")
print(f"deep FPS (VLM, amortised over stride): {deep_fps:7.1f}   "
          f"({1000 * t_caption / n:.1f} ms/frame amortised)")
print(f"end-to-end FPS          : {e2e_fps:7.1f}   "
          f"({1000 * t_e2e / n:.1f} ms/frame)")
print("\nReport shallow and deep FPS separately (challenge convention); "
      "deep depends on caption_stride and object density, so state the clip used.")

